In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
from pathlib import Path

import cv2
import joblib
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image

from torchvision import models
from torchvision import transforms

from ultralytics import YOLO

from scipy.stats import entropy

from tqdm import tqdm

### Configuration & Paths

In [2]:
# ==========================
# CONFIGURATION
# ==========================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TEST_PATH = Path("data/test")

MODEL_PATH = "best_model.pth"
XGB_MODEL_PATH = "best_xgboost_model.pkl"

print(f"Device : {DEVICE}")
print(f"Test Images : {len(list(TEST_PATH.glob('*')))}")

Device : cuda
Test Images : 300


In [3]:
# ==========================
# IMAGE TRANSFORM
# ==========================

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Transform Loaded Successfully!")

Transform Loaded Successfully!


### Load ResNet18 & Create Feature Extractor

In [4]:
# ==========================
# LOAD RESNET18
# ==========================

model = models.resnet18(weights=None)

# Replace FC Head (same as training)
model.fc = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(model.fc.in_features, 3)
)

# Load trained weights
model.load_state_dict(
    torch.load(MODEL_PATH, map_location=DEVICE)
)

model = model.to(DEVICE)
model.eval()

print("ResNet Model Loaded Successfully!")

ResNet Model Loaded Successfully!


In [5]:
# ==========================
# FEATURE EXTRACTOR
# ==========================

feature_extractor = nn.Sequential(
    *list(model.children())[:-1]
)

feature_extractor = feature_extractor.to(DEVICE)
feature_extractor.eval()

print("ResNet Feature Extractor Ready!")

ResNet Feature Extractor Ready!


### Load YOLOv8 & Register Feature Hook

In [7]:
# ==========================
# LOAD YOLO MODEL
# ==========================

yolo_model = YOLO("yolov8n.pt")

print("YOLO Model Loaded Successfully!")

YOLO Model Loaded Successfully!


### Register the Layer 21 Hook

In [8]:
# ==========================
# YOLO FEATURE HOOK
# ==========================

feature_maps = {}

def hook_fn(module, input, output):
    feature_maps["layer21"] = output

hook = yolo_model.model.model[21].register_forward_hook(hook_fn)

print("YOLO Layer 21 Hook Registered!")

YOLO Layer 21 Hook Registered!


### ResNet Feature Extraction (512 Features)

In [10]:
# ==========================
# RESNET FEATURE EXTRACTION
# ==========================

def extract_resnet_features(image_path):

    image = Image.open(image_path).convert("RGB")

    image = transform(image)

    image = image.unsqueeze(0).to(DEVICE)

    with torch.no_grad():

        features = feature_extractor(image)

        features = features.squeeze()

        features = features.cpu().numpy()

    return features

### YOLO Feature Extraction (256 Features)

In [11]:
# ==========================
# YOLO FEATURE EXTRACTION
# ==========================

def extract_yolo_embedding(image_path):

    image = cv2.imread(image_path)

    if image is None:
        return np.zeros(256, dtype=np.float32)

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    with torch.no_grad():
        yolo_model.predict(image, verbose=False)

    fmap = feature_maps["layer21"]

    embedding = F.adaptive_avg_pool2d(fmap, (1, 1))

    embedding = embedding.squeeze().cpu().numpy()

    return embedding

In [12]:
sample_image = next(TEST_PATH.glob("*"))

print("Sample Image :", sample_image)

resnet_feature = extract_resnet_features(str(sample_image))
yolo_feature = extract_yolo_embedding(str(sample_image))

print("ResNet Shape :", resnet_feature.shape)
print("YOLO Shape   :", yolo_feature.shape)

Sample Image : data/test/62ad9097-137f-4d61-bcd4-6d249713f921.png
ResNet Shape : (512,)
YOLO Shape   : (256,)


### Handcrafted Feature Extraction

In [13]:
# ==========================
# HANDCRAFTED FEATURES
# ==========================

def extract_handcrafted_features(image_path):

    image = cv2.imread(image_path)

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)

    features = {}

    # Gray Statistics
    features["mean_gray"] = np.mean(gray)
    features["median_gray"] = np.median(gray)
    features["std_gray"] = np.std(gray)
    features["min_gray"] = np.min(gray)
    features["max_gray"] = np.max(gray)
    features["contrast"] = np.std(gray)

    # Histogram Entropy
    hist = cv2.calcHist([gray], [0], None, [256], [0, 256])
    hist = hist.ravel()
    hist = hist / hist.sum()

    features["entropy"] = entropy(hist)

    # HSV
    features["h_mean"] = hsv[:, :, 0].mean()
    features["s_mean"] = hsv[:, :, 1].mean()
    features["v_mean"] = hsv[:, :, 2].mean()

    features["h_std"] = hsv[:, :, 0].std()
    features["s_std"] = hsv[:, :, 1].std()
    features["v_std"] = hsv[:, :, 2].std()

    # LAB
    features["l_mean"] = lab[:, :, 0].mean()
    features["a_mean"] = lab[:, :, 1].mean()
    features["b_mean"] = lab[:, :, 2].mean()

    features["l_std"] = lab[:, :, 0].std()
    features["a_std"] = lab[:, :, 1].std()
    features["b_std"] = lab[:, :, 2].std()

    # Edge Density
    edges = cv2.Canny(gray, 100, 200)
    features["edge_density"] = np.mean(edges > 0)

    # Sharpness
    features["laplacian_var"] = cv2.Laplacian(
        gray,
        cv2.CV_64F
    ).var()

    # Bright/Dark Pixels
    features["bright_ratio"] = np.mean(gray > 200)
    features["dark_ratio"] = np.mean(gray < 50)

    return features

In [14]:
sample_handcrafted = extract_handcrafted_features(str(sample_image))

print("Number of handcrafted features:", len(sample_handcrafted))
print(sample_handcrafted)

Number of handcrafted features: 23
{'mean_gray': np.float64(79.88200759887695), 'median_gray': np.float64(83.0), 'std_gray': np.float64(33.253363611389084), 'min_gray': np.uint8(0), 'max_gray': np.uint8(196), 'contrast': np.float64(33.253363611389084), 'entropy': np.float32(4.7684994), 'h_mean': np.float64(30.40499496459961), 's_mean': np.float64(93.3339958190918), 'v_mean': np.float64(86.15240478515625), 'h_std': np.float64(16.369823528635802), 's_std': np.float64(46.994922201153685), 'v_std': np.float64(33.40464050149906), 'l_mean': np.float64(86.74224090576172), 'a_mean': np.float64(124.77056884765625), 'b_mean': np.float64(141.47240447998047), 'l_std': np.float64(36.284672182223275), 'a_std': np.float64(4.262978469428021), 'b_std': np.float64(5.157089765751702), 'edge_density': np.float64(0.009098052978515625), 'laplacian_var': np.float64(219.2656243944075), 'bright_ratio': np.float64(0.0), 'dark_ratio': np.float64(0.18491744995117188)}


In [15]:
# ==========================
# LOAD XGBOOST MODEL
# ==========================

xgb_model = joblib.load(XGB_MODEL_PATH)

print("XGBoost Model Loaded Successfully!")
print(xgb_model)

XGBoost Model Loaded Successfully!
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device='cuda', early_stopping_rounds=None,
              enable_categorical=True, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=-1, num_class=3, ...)


In [16]:
# ==========================
# GENERATE TEST FEATURES
# ==========================

test_records = []

image_paths = sorted(TEST_PATH.glob("*"))

for image_path in tqdm(image_paths):

    # ResNet Features (512)
    resnet_features = extract_resnet_features(str(image_path))

    # YOLO Features (256)
    yolo_features = extract_yolo_embedding(str(image_path))

    # Handcrafted Features (23)
    handcrafted = extract_handcrafted_features(str(image_path))

    # Combine all features
    features = np.concatenate([
        resnet_features,
        yolo_features,
        np.array(list(handcrafted.values()), dtype=np.float32)
    ])

    test_records.append({
        "id": image_path.stem,
        "features": features
    })

print("Total Test Images :", len(test_records))
print("Feature Shape :", test_records[0]["features"].shape)

100%|██████████| 300/300 [00:23<00:00, 12.71it/s]

Total Test Images : 300
Feature Shape : (791,)


In [17]:
# ==========================
# PREDICT TEST LABELS
# ==========================

X_test = np.vstack([
    record["features"]
    for record in test_records
])

predictions = xgb_model.predict(X_test)

print("Prediction Shape :", predictions.shape)
print("First 10 Predictions :", predictions[:10])

Prediction Shape : (300,)
First 10 Predictions : [1 0 0 0 2 1 0 1 1 1]


In [18]:
# ==========================
# CREATE SUBMISSION FILE
# ==========================

submission = pd.DataFrame({
    "id": [record["id"] for record in test_records],
    "label": predictions.astype(int)
})

submission.to_csv("submission.csv", index=False)

print(submission.head())
print()
print("submission.csv saved successfully!")

                                     id  label
0  0021e90d-0114-491a-a552-d9ec3d391219      1
1  01061dcb-4112-47b3-9646-acc98fea3d12      0
2  01250393-88e5-431e-9dd6-b84d766bfb61      0
3  026987e3-a347-478d-9bfe-a5795e8e7bd0      0
4  02893c44-a9a0-4da9-a0cb-c4c514cee2fd      2

submission.csv saved successfully!


In [19]:
submission = pd.read_csv("submission.csv")

print(submission.shape)
print(submission.columns.tolist())
print(submission["label"].value_counts())

(300, 2)
['id', 'label']
label
2    104
0    101
1     95
Name: count, dtype: int64
